Cell 1 | markdown
# Cell Annotation — Deep Learning Approaches

> **Goal:** Use Leiden cluster labels and preliminary atlas annotations (from CellTypist) as a starting point, then re-annotate cells and predict marker genes.

## This notebook has two parts

| | Part | What it contains | When to use |
|---|---|---|---|
| **Part 1** | Reference-based deep-learning approaches | 3 supervised classifiers trained on CellTypist labels (retrain CellTypist · scANVI · custom NN) + their consensus | you have a **decent matching atlas** |
| **Part 2** | Reference-free & limited-atlas methods | reject option · scType · AUCell · marker discovery (NS-Forest/Tau/COSG) · zero-shot · cross-family consensus | the atlas is **thin, noisy, mismatched, or absent** |

> ⚠️ **The two parts are alternative menus, not a sequence.** Part 1's three approaches all train on the same
> CellTypist labels, so they share the same blind spot when the atlas is limited. Part 2's methods do **not**
> need a trustworthy atlas. The most defensible annotation combines **one method from each part** and treats
> cells where they disagree as candidate novel cell types. This applies to **any tissue**.

## Part 1 — three approaches at a glance

| | Approach | Complexity | Gives explicit markers | Handles unlabeled cells | Improves embedding |
|---|---|---|---|---|---|
| 1 | Retrain CellTypist | Low | ✓ Yes | ✗ | ✗ |
| 2 | scANVI (semi-supervised VAE) | Medium | Indirect | ✓ Yes | ✓ Yes |
| 3 | Custom neural network on scVI latent | High | Only with SHAP | ✗ | ✗ |

## Data flow (Part 1)

```
CellTypist atlas annotation  ──────────────────┐
(adata.obs["celltypist:{model}"])               │
                                                ▼
Leiden clusters                        training labels (high-conf cells)
(adata.obs["scvi_leiden"])                      │
                                                ▼
scVI latent embedding              ──► train dataset-specific model
(adata.obsm["X_emb"])  30-dim              │
                                           ▼
                                  adata.obs["consensus_celltype"]
```

## Input required
- `adata` with:
  - `adata.X`: raw count matrix
  - `adata.obsm["X_emb"]`: 30-dim scVI latent embedding from Step 3.2
  - `adata.obs["scvi_leiden"]`: Leiden cluster labels from Step 4.3
  - `adata.obs["celltypist:{model_name}"]`: preliminary labels from Step 4.5 (CellTypist)
  - `adata.obs["celltypist:{model_name}:conf"]`: confidence scores from Step 4.5

In [ ]:
from pathlib import Path
import sys

# Run Jupyter with this project folder as its working directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "project_paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "project_paths.py").is_file():
    raise RuntimeError("Set the notebook working directory to the project folder.")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from project_paths import atlas_path, data_path


Cell 2 | markdown
## Setup — Load data and identify CellTypist columns

In [ ]:
# Cell 3 | code
import json
import re
from pathlib import Path

# Run this cell anytime after inserting/deleting/reordering cells.
# It rewrites visible "Cell N | ..." labels across the notebook.
NOTEBOOK_PATH = PROJECT_ROOT / "scripts" / "cell_annotation_DL_v2_3.ipynb"

with NOTEBOOK_PATH.open("r", encoding="utf-8") as f:
    nb = json.load(f)

for idx, cell in enumerate(nb.get("cells", []), start=1):
    src = cell.get("source", [])
    if isinstance(src, str):
        src = [src]

    normalized = [line.rstrip("\n") for line in src]

    # Remove old auto-label if present.
    if normalized and re.match(r"^(#\s*)?Cell\s+\d+\s*\|", normalized[0].strip()):
        normalized = normalized[1:]

    label = f"Cell {idx} | markdown" if cell.get("cell_type") == "markdown" else f"# Cell {idx} | code"
    cell["source"] = [label] + normalized

with NOTEBOOK_PATH.open("w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)
    f.write("\n")

print(f"Renumbered {len(nb.get('cells', []))} cells in: {NOTEBOOK_PATH}")
print("Re-open the notebook tab (or reload from disk) to see updated labels.")

In [ ]:
# Cell 4 | code
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# Fast mode: keep lightweight output for repeated runs
VERBOSE_SUMMARY = False

# Load your CellTypist-annotated h5ad from Step 5
h5ad_path = str(data_path('scdownstream/5.pre_hvg_annotation/merged_celltypist_immune_low.h5ad'))
adata = sc.read_h5ad(h5ad_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")

# Identify CellTypist label columns
celltypist_label_cols = [
    c for c in adata.obs.columns
    if c.startswith("celltypist:") and not c.endswith(":conf")
]
if not celltypist_label_cols:
    raise ValueError("No CellTypist label columns found in adata.obs")

# Pick the model to use as preliminary annotation
CELLTYPIST_COL = celltypist_label_cols[0]   # adjust index or set explicitly
CELLTYPIST_CONF_COL = CELLTYPIST_COL + ":conf"
CONF_THRESHOLD = 0.7

if CELLTYPIST_CONF_COL not in adata.obs.columns:
    raise ValueError(f"Missing confidence column: {CELLTYPIST_CONF_COL}")

print(f"Using: {CELLTYPIST_COL}")
print(f"Confidence column: {CELLTYPIST_CONF_COL}")
print(f"Confidence threshold: {CONF_THRESHOLD}")

# Optional detailed summary (can be slow on large datasets)
if VERBOSE_SUMMARY:
    print("\nAvailable CellTypist label columns:")
    for col in celltypist_label_cols:
        conf_col = col + ":conf"
        n_types = adata.obs[col].nunique()
        n_high_conf = (adata.obs[conf_col] > CONF_THRESHOLD).sum() if conf_col in adata.obs else 0
        print(f"  {col}")
        print(f"    Cell types : {n_types}")
        print(f"    High-conf (>{CONF_THRESHOLD}) cells : {n_high_conf} / {adata.n_obs}")

    print("\nLabel distribution:")
    print(adata.obs[CELLTYPIST_COL].value_counts())

Cell 5 | markdown
---
# ═════════  PART 1  ════════════
# Reference-Based Deep-Learning Approaches
*(require a matching atlas — all three train on the CellTypist labels)*

Three supervised classifiers, then their consensus. Each re-annotates the cells and, for Approaches 1 and 3,
predicts markers. **If your atlas is limited, read Part 1 as a baseline and rely on Part 2.**

Cell 6 | markdown
---
## Approach 1 — Retrain CellTypist on your data

**Concept:** CellTypist's classifier is a logistic regression with weight matrix $W \in \mathbb{R}^{C \times G}$ (cell types × genes). By training it on YOUR cells with YOUR atlas labels, it learns which gene combinations are discriminative specifically in your dataset — not just in the reference atlas.

**How the marker learning works:**

When `feature_selection=True`, CellTypist runs a two-pass training:
1. **Pass 1:** Train on all $G$ genes → compute coefficient $|w_{cg}|$ for every gene $g$ in every cell type $c$
2. **Pass 2:** Keep only `top_genes` with highest $|w_{cg}|$ → retrain on that sparse set

The genes that survive Pass 2 are your **dataset-specific markers** — the genes that best separate your cell types in your data.

**Pros:**
- Fast to run
- Gives an explicit ranked list of discriminative marker genes per cell type
- Produces a portable `.pkl` model you can reuse on new samples

**Cons:**
- Each cell needs a label — cells below the confidence threshold are excluded from training (not used at all)
- Linear model only — cannot capture complex nonlinear gene interactions

In [ ]:
# Cell 7 | code
import os
import inspect
import importlib
import contextlib
import numpy as np
import celltypist
from celltypist import models as ct_models
from sklearn.linear_model import LogisticRegression as SkLogisticRegression

# ── Compatibility patch for scikit-learn >= 1.9 ─────────────────────────────
# CellTypist 1.7.1 passes `multi_class=...` into LogisticRegression.
# In scikit-learn 1.9+, that kwarg was removed.
ct_train_module = importlib.import_module("celltypist.train")
if "multi_class" not in inspect.signature(SkLogisticRegression).parameters:
    def _logreg_compat(*args, multi_class=None, **kwargs):
        return SkLogisticRegression(*args, **kwargs)

    # Patch the exact symbol used inside celltypist.train._LRClassifier
    ct_train_module.LogisticRegression = _logreg_compat
    print("Applied CellTypist compatibility patch for scikit-learn >= 1.9")

# ── Output directory for Approach 1 artifacts ─────────────────────────────────
OUTPUT_DIR = str(data_path('cell_annotation_optimize/1.celltypist_retrain'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Step 1: prepare normalized copy for CellTypist training ──────────────────
adata_train_ct = adata.copy()
sc.pp.normalize_per_cell(adata_train_ct, counts_per_cell_after=1e4)
sc.pp.log1p(adata_train_ct)

# keep only high-confidence cells as training examples
high_conf_mask = adata_train_ct.obs[CELLTYPIST_CONF_COL] > CONF_THRESHOLD
adata_train_ct = adata_train_ct[high_conf_mask].copy()
print(f"Training cells (conf > {CONF_THRESHOLD}): {adata_train_ct.n_obs} / {adata.n_obs}")
print(f"Cell types in training set: {adata_train_ct.obs[CELLTYPIST_COL].nunique()}")

# ── Step 2: train the dataset-specific model ──────────────────────────────────
retrained_model = celltypist.train(
    adata_train_ct,
    labels=CELLTYPIST_COL,       # column with preliminary atlas labels
    n_jobs=8,
    max_iter=200,
    feature_selection=True,      # learn discriminative marker genes
    top_genes=300,               # number of top markers retained per model
    check_expression=False       # skip expression check (already normalized)
)

RETRAINED_MODEL_PATH = os.path.join(OUTPUT_DIR, "retrained_celltypist.pkl")
retrained_model.write(RETRAINED_MODEL_PATH)
print(f"Model saved: {RETRAINED_MODEL_PATH}")

# ── Step 2.1: explicit model source selection ─────────────────────────────────
ATLAS_MODEL_PATH = str(atlas_path())

# Choose one of: "retrained_memory", "retrained_pkl", "atlas"
ANNOTATE_MODEL_SOURCE = "retrained_memory"

if ANNOTATE_MODEL_SOURCE == "retrained_memory":
    ANNOTATE_MODEL = retrained_model
    print("Annotation model source: retrained model object (current run)")
elif ANNOTATE_MODEL_SOURCE == "retrained_pkl":
    ANNOTATE_MODEL = RETRAINED_MODEL_PATH
    print(f"Annotation model source: retrained model file -> {RETRAINED_MODEL_PATH}")
elif ANNOTATE_MODEL_SOURCE == "atlas":
    ANNOTATE_MODEL = ATLAS_MODEL_PATH
    print(f"Annotation model source: atlas model file -> {ATLAS_MODEL_PATH}")
else:
    raise ValueError(
        "ANNOTATE_MODEL_SOURCE must be one of: retrained_memory, retrained_pkl, atlas"
    )

# Hard guard: never allow implicit model-catalog fallback.
if isinstance(ANNOTATE_MODEL, str):
    if not os.path.isabs(ANNOTATE_MODEL):
        raise ValueError(
            f"Model path must be absolute to avoid fallback/download: {ANNOTATE_MODEL}"
        )
    if not os.path.isfile(ANNOTATE_MODEL):
        raise FileNotFoundError(
            f"Model file not found (refusing fallback/download): {ANNOTATE_MODEL}"
        )
elif not hasattr(ANNOTATE_MODEL, "classifier"):
    raise TypeError(
        "ANNOTATE_MODEL must be a CellTypist model object or absolute .pkl path"
    )

print(f"Resolved annotation model type: {type(ANNOTATE_MODEL)}")
if isinstance(ANNOTATE_MODEL, str):
    print(f"Resolved annotation model path: {ANNOTATE_MODEL}")

@contextlib.contextmanager
def block_celltypist_downloads():
    original = getattr(ct_models, "download_models", None)

    def _blocked_download(*args, **kwargs):
        raise RuntimeError(
            "Blocked CellTypist built-in model download. "
            "Pass a valid in-memory model object or existing absolute .pkl path."
        )

    if original is not None:
        ct_models.download_models = _blocked_download
    try:
        yield
    finally:
        if original is not None:
            ct_models.download_models = original

# ── Step 3: inspect the learned dataset-specific markers ─────────────────────
coef_df = pd.DataFrame(
    retrained_model.classifier.coef_,
    index=retrained_model.cell_types,
    columns=retrained_model.features
)

print("\nTop 10 positive marker genes per cell type:")
marker_dict = {}
for ct in coef_df.index:
    top_markers = coef_df.loc[ct].nlargest(10).index.tolist()
    marker_dict[ct] = top_markers
    print(f"\n  {ct}:")
    print(f"    {top_markers}")

# Save marker table for downstream review
marker_table = pd.DataFrame(
    {ct: coef_df.loc[ct].nlargest(50).index.tolist() for ct in coef_df.index}
)
marker_path = os.path.join(OUTPUT_DIR, "retrained_celltypist_top50_markers.csv")
marker_table.to_csv(marker_path, index=False)
print(f"Top-marker table saved: {marker_path}")

# ── Step 4: re-annotate ALL cells with the selected model ─────────────────────
adata_full_ct = adata.copy()
sc.pp.normalize_per_cell(adata_full_ct, counts_per_cell_after=1e4)
sc.pp.log1p(adata_full_ct)

# Majority voting needs a cluster column. Prefer scvi_leiden if available.
cluster_col = None
for c in ["scvi_leiden", "leiden_0.5", "leiden_1.0", "leiden"]:
    if c in adata.obs.columns:
        cluster_col = c
        break

with block_celltypist_downloads():
    if cluster_col is None:
        print("No cluster column found (scvi_leiden/leiden_*).")
        print("Running per-cell CellTypist annotation without majority voting.")
        pred_ct = celltypist.annotate(
            adata_full_ct,
            model=ANNOTATE_MODEL,
            majority_voting=False,
        )
    else:
        print(f"Using cluster column for majority voting: {cluster_col}")
        pred_ct = celltypist.annotate(
            adata_full_ct,
            model=ANNOTATE_MODEL,
            majority_voting=True,
            over_clustering=adata.obs[cluster_col],
        )

pred_ct_adata = pred_ct.to_adata()

# Majority-voting label may be missing when majority_voting=False.
if "majority_voting" in pred_ct_adata.obs.columns:
    adata.obs["approach1_celltype"] = pred_ct_adata.obs["majority_voting"]
else:
    adata.obs["approach1_celltype"] = pred_ct_adata.obs["predicted_labels"]

adata.obs["approach1_celltype_percell"] = pred_ct_adata.obs["predicted_labels"]
adata.obs["approach1_conf"] = pred_ct_adata.obs["conf_score"]

print("\nApproach 1 prediction distribution:")
print(adata.obs["approach1_celltype"].value_counts())

# Save per-cell predictions
pred_cols = ["approach1_celltype", "approach1_celltype_percell", "approach1_conf"]
pred_path = os.path.join(OUTPUT_DIR, "approach1_predictions.csv")
adata.obs[pred_cols].to_csv(pred_path)
print(f"Prediction table saved: {pred_path}")

# ── Step 5: ensure UMAP coordinates exist for plotting ───────────────────────
if "X_umap" not in adata.obsm:
    print("X_umap not found; computing UMAP for visualization...")
    adata_plot = adata.copy()
    if "X_emb" in adata_plot.obsm:
        n_pcs_vis = min(30, adata_plot.obsm["X_emb"].shape[1])
        sc.pp.neighbors(adata_plot, use_rep="X_emb", n_pcs=n_pcs_vis)
    else:
        # Fallback: build a quick PCA-based graph if X_emb is unavailable.
        sc.pp.normalize_total(adata_plot, target_sum=1e4)
        sc.pp.log1p(adata_plot)
        n_pcs_vis = min(30, max(2, adata_plot.n_vars - 1))
        sc.pp.pca(adata_plot, n_comps=n_pcs_vis)
        sc.pp.neighbors(adata_plot, n_pcs=n_pcs_vis)
    sc.tl.umap(adata_plot, random_state=0)
    adata.obsm["X_umap"] = adata_plot.obsm["X_umap"].copy()

# Save re-annotated AnnData
annotated_h5ad_path = os.path.join(OUTPUT_DIR, "approach1_reannotated.h5ad")
adata.write_h5ad(annotated_h5ad_path)
print(f"Re-annotated AnnData saved: {annotated_h5ad_path}")

# ── Step 6: visualize (layout tuned to avoid overlap) ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(26, 6), constrained_layout=True)

sc.pl.umap(
    adata,
    color=CELLTYPIST_COL,
    title="Atlas labels (input)",
    ax=axes[0],
    show=False,
    legend_loc="right margin",
    legend_fontsize=7,
)

sc.pl.umap(
    adata,
    color="approach1_celltype",
    title="Approach 1 (majority)",
    ax=axes[1],
    show=False,
    legend_loc="right margin",
    legend_fontsize=7,
)

sc.pl.umap(
    adata,
    color="approach1_conf",
    title="Approach 1 confidence",
    ax=axes[2],
    show=False,
)

plot_path = os.path.join(OUTPUT_DIR, "approach1_umap.png")
plt.savefig(plot_path, dpi=180, bbox_inches="tight")
print(f"UMAP plot saved: {plot_path}")
plt.show()

Cell 8 | markdown
---
## Approach 2 — Custom Semi-Supervised NB-VAE (scANVI-style logic)

**Concept:** Keep the same semi-supervised logic as scANVI (labeled + unlabeled training), but replace ZINB/scANVI latent training with a custom **NB-VAE** backbone. Labeled cells guide the classifier head, while all cells contribute to reconstruction and latent regularization.

**Architecture:**

```
adata.X (raw counts)
      │
      ▼
 NB-VAE encoder  →  z_i ∈ ℝ³⁰
        │                 │
        │                 └──► classifier  →  P(cell type | z_i)   (labeled cells)
        └──► NB decoder   →  reconstructed counts                  (all cells)
```

**Training objective:**

$$
\mathcal{L} = \underbrace{-\log p_{NB}(x\mid z)}_{\text{reconstruction, all cells}} +
\underbrace{\beta\,\mathrm{KL}(q(z\mid x)\|p(z))}_{\text{latent regularization, all cells}} +
\underbrace{\alpha\,\mathrm{CE}(\hat y, y)}_{\text{classification, labeled cells only}}
$$

Where low-confidence cells are set to `"Unknown"` and excluded from the classification term.

**Why this variant:**
- Keeps scANVI-like semi-supervised label transfer logic
- Uses NB likelihood (faster than ZINB in your tests)
- Produces a learned latent embedding (`X_scANVI`) for downstream UMAP/clustering comparison

**Pros:**
- Faster than ZINB-based scANVI in many CPU workflows
- Keeps unlabeled-cell handling and semi-supervised learning behavior
- Fully customizable architecture and loss weights

**Cons:**
- Not the exact scvi-tools scANVI implementation
- Requires tuning of classification weight and early stopping
- No built-in scvi-tools training callbacks

In [ ]:
# Cell 9 | code
# Run Cell 4 first before running this cell
import os
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.sparse import issparse
from sklearn.model_selection import train_test_split

# ── Disable torch.compile to avoid circular imports ──────────────────
torch.compile = lambda x: x  # No-op: avoids loading torch._inductor


# ── Compatibility patch for broken torch._utils in notebook kernels ──────────
# Some notebook kernels hold stale torch imports that lack torch._utils.
# Force-reimport to fix the issue; fallback to patching Adam if needed.
try:
    from torch import _utils
except (ImportError, AttributeError):
    print("Warning: torch._utils not available; attempting to patch...")
    import sys
    import importlib
    if 'torch' in sys.modules:
        # Reload torch to get a fresh import
        try:
            importlib.reload(sys.modules['torch'])
        except Exception:
            pass
    # Try again
    try:
        from torch import _utils
        print("torch._utils recovered after reload")
    except (ImportError, AttributeError):
        # Last resort: create a minimal _utils stub
        class _UtilsStub:
            pass
        sys.modules['torch._utils'] = _UtilsStub()
        torch._utils = _UtilsStub()
        print("torch._utils stubbed; proceeding with training")

# ── Output directory for Approach 2 artifacts ─────────────────────────────────
OUTPUT_DIR = str(data_path('cell_annotation_optimize/2.customNB-VAE_scANVI_style'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cpu":
    torch.set_num_threads(os.cpu_count())
    print(f"CPU training: using all {os.cpu_count()} threads (torch.set_num_threads)")

# ── Guard: this cell needs adata/CELLTYPIST_* from Cell 4 in the current kernel session ──
_required = ["adata", "CELLTYPIST_COL", "CELLTYPIST_CONF_COL", "CONF_THRESHOLD"]
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise NameError(
        f"Missing {_missing} — run Cell 4 (Setup) in this kernel session before running this cell."
    )

# ── Step 1: load HVG raw-count object used by scVI/scANVI workflow ───────────
scvi_h5ad = str(data_path('scdownstream/6.integrate/6.1_hvgs/merged_hvgs.h5ad'))
adata_scvi = sc.read_h5ad(scvi_h5ad)
print(f"Loaded: {adata_scvi.n_obs} cells x {adata_scvi.n_vars} genes")

# ── Step 2: build semi-supervised labels (same logic as scANVI) ──────────────
# Convert labels to plain string array (avoid categorical dtype issues)
label_values = adata.obs[CELLTYPIST_COL].reindex(adata_scvi.obs_names).values
label_values = np.array([str(v) for v in label_values], dtype=object)

# Check confidence and mark low-conf cells as Unknown
conf_values = adata.obs[CELLTYPIST_CONF_COL].reindex(adata_scvi.obs_names).values
low_conf_mask = (pd.isna(conf_values)) | (conf_values < CONF_THRESHOLD)
label_values[low_conf_mask] = "Unknown"

# Assign as plain object array (not categorical)
adata_scvi.obs["train_labels"] = pd.Series(label_values, index=adata_scvi.obs_names, dtype=object)

n_labeled = (adata_scvi.obs["train_labels"] != "Unknown").sum()
n_unlabeled = (adata_scvi.obs["train_labels"] == "Unknown").sum()
print(f"Labeled cells   : {n_labeled}")
print(f'Unlabeled cells ("Unknown"): {n_unlabeled}')

# ── Step 3: prepare tensors ───────────────────────────────────────────────────
X = adata_scvi.X.toarray().astype(np.float32) if issparse(adata_scvi.X) else adata_scvi.X.astype(np.float32)
lib_size = X.sum(axis=1, keepdims=True).clip(1)
log_lib = np.log(lib_size).squeeze(1)

labels_str = adata_scvi.obs["train_labels"].values
labeled_mask = labels_str != "Unknown"

label_classes = sorted(pd.Series(labels_str[labeled_mask]).unique().tolist())
label_to_idx = {c: i for i, c in enumerate(label_classes)}
y = np.full(X.shape[0], -1, dtype=np.int64)
y[labeled_mask] = np.array([label_to_idx[v] for v in labels_str[labeled_mask]], dtype=np.int64)

# Keep validation stratified on labeled cells only
labeled_idx = np.where(labeled_mask)[0]
n_labeled_total = len(labeled_idx)
n_classes = len(label_classes)

# Check if stratification is feasible: EACH class needs at least 2 samples
y_labeled = y[labeled_idx]
unique_labels, class_counts = np.unique(y_labeled, return_counts=True)
min_class_count = class_counts.min()
test_size = 0.1

# For stratification to work, we need at least 2 samples per class in BOTH train and test
# This requires each class to have at least 2/test_size samples (e.g., 20 for test_size=0.1)
min_needed = max(2, int(np.ceil(2 / test_size)))

can_stratify = (min_class_count >= min_needed)
if not can_stratify:
    print(f"Warning: stratification not feasible.")
    print(f"  Min class count: {min_class_count}")
    print(f"  Needed for stratified split (test_size={test_size}): {min_needed}")
    print("Using non-stratified split instead.")
    _, val_lab_idx = train_test_split(
        labeled_idx,
        test_size=min(test_size, 0.5),  # cap at 50% to keep some training data
        random_state=0,
        stratify=None,
    )
else:
    _, val_lab_idx = train_test_split(
        labeled_idx,
        test_size=test_size,
        random_state=0,
        stratify=y[labeled_idx],
    )

all_idx = np.arange(X.shape[0])
val_mask = np.zeros(X.shape[0], dtype=bool)
val_mask[val_lab_idx] = True
train_idx = all_idx[~val_mask]
val_idx = val_lab_idx

print(f"Train cells: {len(train_idx)}")
print(f"Val labeled cells: {len(val_idx)}")
print(f"Classes: {len(label_classes)}")

# ── Step 4: model (NB-VAE + classifier head) ─────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, n_input, n_hidden, n_latent):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
            nn.Linear(n_hidden[0], n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(n_hidden[1], n_latent)
        self.fc_log_var = nn.Linear(n_hidden[1], n_latent)

    def forward(self, x):
        h = self.net(torch.log1p(x))
        return self.fc_mu(h), self.fc_log_var(h)


class Decoder(nn.Module):
    def __init__(self, n_latent, n_hidden, n_output):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_latent, n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
            nn.Linear(n_hidden[1], n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
        )
        self.fc_scale = nn.Linear(n_hidden[0], n_output)
        self.log_theta = nn.Parameter(torch.zeros(n_output))

    def forward(self, z, log_library):
        h = self.net(z)
        px_scale = F.softmax(self.fc_scale(h), dim=-1)
        px_mu = px_scale * torch.exp(log_library).unsqueeze(1)
        px_theta = torch.exp(self.log_theta)
        return px_mu, px_theta


class SemiSupervisedNBVAE(nn.Module):
    def __init__(self, n_genes, n_hidden, n_latent, n_classes):
        super().__init__()
        self.encoder = Encoder(n_genes, n_hidden, n_latent)
        self.decoder = Decoder(n_latent, n_hidden, n_genes)
        self.classifier = nn.Sequential(
            nn.Linear(n_latent, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std

    def forward(self, x, log_library):
        mu_z, log_var_z = self.encoder(x)
        z = self.reparameterize(mu_z, log_var_z)
        px_mu, px_theta = self.decoder(z, log_library)
        cls_logits = self.classifier(mu_z)
        return px_mu, px_theta, mu_z, log_var_z, cls_logits

    @torch.no_grad()
    def get_latent(self, x):
        self.eval()
        mu_z, _ = self.encoder(x)
        return mu_z

    @torch.no_grad()
    def predict_labels(self, x):
        self.eval()
        mu_z, _ = self.encoder(x)
        logits = self.classifier(mu_z)
        probs = torch.softmax(logits, dim=1)
        return probs


def nb_log_likelihood(x, mu, theta, eps=1e-8):
    log_theta_mu = torch.log(theta + mu + eps)
    return (
        theta * (torch.log(theta + eps) - log_theta_mu)
        + x * (torch.log(mu + eps) - log_theta_mu)
        + torch.lgamma(x + theta)
        - torch.lgamma(theta)
        - torch.lgamma(x + 1)
    )


def loss_components(x, px_mu, px_theta, mu_z, log_var_z):
    recon = -nb_log_likelihood(x, px_mu, px_theta).sum(dim=1).mean()
    kl = -0.5 * (1 + log_var_z - mu_z.pow(2) - log_var_z.exp()).sum(dim=1).mean()
    return recon, kl


# ── Step 5: train (scANVI-style semi-supervised objective with NB-VAE) ───────
n_latent = 30
n_hidden = [256, 128]
n_epochs = 800
batch_size = 1024  # larger batch = fewer Python-loop iterations per epoch (big win on CPU)
lr = 1e-4
kl_warmup = 50
patience = 30
alpha_cls = 1.0

model = SemiSupervisedNBVAE(
    n_genes=adata_scvi.n_vars,
    n_hidden=n_hidden,
    n_latent=n_latent,
    n_classes=len(label_classes),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
ce_loss = nn.CrossEntropyLoss()

X_t = torch.tensor(X, dtype=torch.float32, device=device)
LL_t = torch.tensor(log_lib, dtype=torch.float32, device=device)
y_t = torch.tensor(y, dtype=torch.long, device=device)

val_idx_t = torch.tensor(val_idx, dtype=torch.long, device=device)

train_losses = []
val_losses = []
best_val = np.inf
no_improve = 0
best_state = None

for epoch in range(1, n_epochs + 1):
    kl_w = min(1.0, epoch / kl_warmup)
    model.train()

    perm = train_idx[np.random.permutation(len(train_idx))]
    epoch_loss = 0.0

    for start in range(0, len(perm), batch_size):
        bidx_np = perm[start:start + batch_size]
        bidx = torch.tensor(bidx_np, dtype=torch.long, device=device)

        x_b = X_t[bidx]
        ll_b = LL_t[bidx]
        y_b = y_t[bidx]

        px_mu, px_theta, mu_z, lv_z, logits = model(x_b, ll_b)
        recon, kl = loss_components(x_b, px_mu, px_theta, mu_z, lv_z)

        labeled_b = y_b >= 0
        if labeled_b.any():
            cls = ce_loss(logits[labeled_b], y_b[labeled_b])
        else:
            cls = torch.tensor(0.0, device=device)

        loss = recon + kl_w * kl + alpha_cls * cls

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        epoch_loss += loss.item() * len(bidx_np)

    train_losses.append(epoch_loss / len(train_idx))

    model.eval()
    with torch.no_grad():
        x_v = X_t[val_idx_t]
        ll_v = LL_t[val_idx_t]
        y_v = y_t[val_idx_t]

        px_mu_v, px_theta_v, mu_z_v, lv_z_v, logits_v = model(x_v, ll_v)
        recon_v, kl_v = loss_components(x_v, px_mu_v, px_theta_v, mu_z_v, lv_z_v)
        cls_v = ce_loss(logits_v, y_v)
        val_loss = (recon_v + kl_v + alpha_cls * cls_v).item()

    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1

    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | train={train_losses[-1]:.4f} | val={val_loss:.4f} | kl_w={kl_w:.2f}")

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"Best val loss: {best_val:.4f}")

# ── Step 5.1: plot training vs validation loss curve ────────────────────
fig_loss, ax_loss = plt.subplots(figsize=(8, 5))
ax_loss.plot(train_losses, label="train")
ax_loss.plot(val_losses, label="val")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss")
ax_loss.set_title("Approach 2 (NB-VAE) — training vs validation loss")
ax_loss.legend()
loss_curve_path = os.path.join(OUTPUT_DIR, "approach2_training_curve.png")
plt.savefig(loss_curve_path, dpi=150, bbox_inches="tight")
print(f"Training curve saved: {loss_curve_path}")
plt.show()

# ── Step 6: predictions + latent embedding ────────────────────────────────────
model.eval()
with torch.no_grad():
    Z = model.get_latent(X_t).cpu().numpy()
    probs = model.predict_labels(X_t).cpu().numpy()

pred_idx = probs.argmax(axis=1)
pred_conf = probs.max(axis=1)
pred_labels = np.array([label_classes[i] for i in pred_idx], dtype=object)

adata_scvi.obsm["X_scANVI"] = Z
adata_scvi.obs["approach2_celltype"] = pred_labels
adata_scvi.obs["approach2_conf"] = pred_conf

adata.obsm["X_scANVI"] = pd.DataFrame(Z, index=adata_scvi.obs_names).reindex(adata.obs_names).values
adata.obs["approach2_celltype"] = adata_scvi.obs["approach2_celltype"].reindex(adata.obs_names)
adata.obs["approach2_conf"] = adata_scvi.obs["approach2_conf"].reindex(adata.obs_names)

print("\nApproach 2 prediction distribution:")
print(adata.obs["approach2_celltype"].value_counts())

n_unknown_pred = (adata.obs["approach2_celltype"] == "Unknown").sum()
print(f"\nSanity check: 'Unknown' entries in approach2_celltype predictions: {n_unknown_pred}")
print("(should be 0 — the classifier never outputs 'Unknown' since it wasn't a training class;")
print(" 'Unknown' only appears in adata_scvi.obs['train_labels'], the pre-training input labels)")

# ── Step 6.1: save predictions ──────────────────────────────────────────────────
pred_cols = ["approach2_celltype", "approach2_conf"]
pred_path = os.path.join(OUTPUT_DIR, "approach2_predictions.csv")
adata.obs[pred_cols].to_csv(pred_path)
print(f"Prediction table saved: {pred_path}")

# ── Step 7: recompute UMAP on NB-VAE latent embedding ────────────────────────
sc.pp.neighbors(adata_scvi, use_rep="X_scANVI", n_pcs=30)
sc.tl.umap(adata_scvi, random_state=0)
adata.obsm["X_umap_scanvi"] = pd.DataFrame(
    adata_scvi.obsm["X_umap"], index=adata_scvi.obs_names
).reindex(adata.obs_names).values
print("New Approach 2 UMAP computed")

# ── Step 8: save updated AnnData objects ──────────────────────────────────────
adata_h5ad_path = os.path.join(OUTPUT_DIR, "approach2_adata_reannotated.h5ad")
adata.write_h5ad(adata_h5ad_path)
print(f"Updated adata saved: {adata_h5ad_path}")

adata_scvi_h5ad_path = os.path.join(OUTPUT_DIR, "approach2_adata_scvi_reannotated.h5ad")
adata_scvi.write_h5ad(adata_scvi_h5ad_path)
print(f"Updated adata_scvi saved: {adata_scvi_h5ad_path}")

# ── Step 9: save model artifact ───────────────────────────────────────────────
model_path = os.path.join(OUTPUT_DIR, "scanvi_nbvae_model.pt")
torch.save(
    {
        "model_state": best_state,
        "label_classes": label_classes,
        "params": {
            "n_latent": n_latent,
            "n_hidden": n_hidden,
            "n_epochs": n_epochs,
            "batch_size": batch_size,
            "lr": lr,
            "kl_warmup": kl_warmup,
            "patience": patience,
            "alpha_cls": alpha_cls,
        },
    },
    model_path,
)
print(f"Model saved: {model_path}")

In [ ]:
# Cell 9.x1 | code
# Approach 2 — visualize: compare scVI UMAP vs custom NB-VAE UMAP
fig, axes = plt.subplots(2, 2, figsize=(20, 10), constrained_layout=True)

# Ensure adata has a UMAP basis for plotting in this cell.
if "X_umap" not in adata.obsm:
    print("X_umap not found in adata; computing UMAP for plotting...")
    adata_plot = adata.copy()
    if "X_emb" in adata_plot.obsm:
        n_pcs_vis = min(30, adata_plot.obsm["X_emb"].shape[1])
        sc.pp.neighbors(adata_plot, use_rep="X_emb", n_pcs=n_pcs_vis)
    elif "X_scANVI" in adata_plot.obsm:
        n_pcs_vis = min(30, adata_plot.obsm["X_scANVI"].shape[1])
        sc.pp.neighbors(adata_plot, use_rep="X_scANVI", n_pcs=n_pcs_vis)
    else:
        sc.pp.normalize_total(adata_plot, target_sum=1e4)
        sc.pp.log1p(adata_plot)
        n_pcs_vis = min(30, max(2, adata_plot.n_vars - 1))
        sc.pp.pca(adata_plot, n_comps=n_pcs_vis)
        sc.pp.neighbors(adata_plot, n_pcs=n_pcs_vis)
    sc.tl.umap(adata_plot, random_state=0)
    adata.obsm["X_umap"] = adata_plot.obsm["X_umap"].copy()
    print("Computed adata X_umap for plotting")

# Detect available cluster column (same logic as Cell 7)
cluster_col = None
for c in ["scvi_leiden", "leiden_0.5", "leiden_1.0", "leiden"]:
    if c in adata.obs.columns:
        cluster_col = c
        break

if cluster_col:
    sc.pl.umap(adata, color=cluster_col, title=f"Leiden ({cluster_col})", ax=axes[0, 0], show=False, legend_loc="right margin", legend_fontsize=7)
else:
    print(f"Warning: no cluster column found. Skipping cluster plot.")
    axes[0, 0].text(0.5, 0.5, "No cluster column found", ha="center", va="center")

sc.pl.umap(adata, color=CELLTYPIST_COL, title="Atlas labels (scVI UMAP)", ax=axes[0, 1], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata_scvi, color="approach2_celltype", title="Approach 2 (NB-VAE UMAP)", ax=axes[1, 0], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata_scvi, color="train_labels", title="Training labels (NB-VAE UMAP)", ax=axes[1, 1], show=False, legend_loc="right margin", legend_fontsize=7)

umap_path = os.path.join(OUTPUT_DIR, "approach2_umap.png")
plt.savefig(umap_path, dpi=180, bbox_inches="tight")
print(f"UMAP plot saved: {umap_path}")
plt.show()

Cell 10 | markdown
---
## Approach 3 — Custom Neural Network on scVI Latent Embedding

**Concept:** Train a 3-layer feedforward neural network directly on the 30-dimensional scVI embedding (`X_emb`). The network learns a nonlinear decision boundary in the latent space to classify cell types.

**Architecture:**

```
X_emb (30-dim)
      │
      ▼
  Linear(30→128) + ReLU + Dropout(0.3)
      │
      ▼
  Linear(128→64) + ReLU + Dropout(0.3)
      │
      ▼
  Linear(64→C)          C = number of cell types
      │
      ▼
  Softmax  →  P(cell type | cell)
```

**Loss function (cross-entropy):**

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i^{\text{true}} \mid x_i^{\text{emb}})$$

Where $x_i^{\text{emb}} \in \mathbb{R}^{30}$ is the scVI latent vector for cell $i$.

**Why train on `X_emb` instead of raw gene expression?**
- `X_emb` is 30-dim — much faster to train than using ~3,000 HVG expression values
- The scVI encoder has already compressed and denoised the expression signal
- The network only needs to learn a classification boundary in the already-structured latent space

**Pros:**
- Can learn nonlinear patterns that logistic regression (Approach 1) cannot
- Fast — 30-dim input trains in seconds
- Fully customizable architecture

**Cons:**
- Does not give explicit marker genes (would need SHAP analysis separately)
- Cells below confidence threshold are excluded from training
- Does not improve the embedding (unlike Approach 2)

In [ ]:
# Cell 11 | code
import os
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ── Output directory for Approach 3 artifacts ─────────────────────────────────
OUTPUT_DIR = str(data_path('cell_annotation_optimize/3.customNN_onXemb'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Step 0: Load X_emb from NBVAE file ────────────────────────────────────────
# Load X_emb from the Step 6.2 NBVAE integration result
adata_nbvae = sc.read_h5ad(str(data_path('scdownstream/6.integrate/6.2.1_NBVAE/merged_nbvae.h5ad')))
adata.obsm["X_emb"] = adata_nbvae.obsm["X_emb"]
print(f"✓ X_emb loaded from NBVAE file: {adata.obsm['X_emb'].shape}")

# ── Step 1: prepare features and labels ───────────────────────────────────────
X_all = adata.obsm["X_emb"].astype(np.float32)   # (N_cells, 30) scVI latent embedding

# only train on high-confidence atlas calls
mask = (adata.obs[CELLTYPIST_CONF_COL] > CONF_THRESHOLD).values

# fit label space on the high-confidence subset only, matching Approaches 1/2 —
# otherwise classes only ever seen with low confidence get an untrainable output slot
le = LabelEncoder()
y_conf = le.fit_transform(adata.obs[CELLTYPIST_COL][mask])
print(f"Cell types ({len(le.classes_)}): {le.classes_.tolist()}")
X_conf = X_all[mask]

# Try stratified split; if some classes have <2 members, fall back to non-stratified
try:
    X_train, X_val, y_train, y_val = train_test_split(
        X_conf, y_conf, test_size=0.2, stratify=y_conf, random_state=0
    )
except ValueError:
    print("Warning: stratification not feasible (some classes have <2 members).")
    print("Using non-stratified split instead.")
    X_train, X_val, y_train, y_val = train_test_split(
        X_conf, y_conf, test_size=0.2, stratify=None, random_state=0
    )
print(f"Training cells  : {len(X_train)}")
print(f"Validation cells: {len(X_val)}")

# ── Step 2: define the network ────────────────────────────────────────────────
class CellTypeNet(nn.Module):
    def __init__(self, n_input, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64),     nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)

n_classes = len(le.classes_)
model_nn  = CellTypeNet(n_input=30, n_classes=n_classes)
optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

X_tr = torch.tensor(X_train)
y_tr = torch.tensor(y_train)
X_vl = torch.tensor(X_val)
y_vl = torch.tensor(y_val)

# ── Step 3: training loop (mini-batch + early stopping, matching Approach 2) ──
n_epochs = 800
batch_size = 256
patience = 30

train_losses = []
val_losses   = []
val_accs     = []
best_val_loss = np.inf
no_improve = 0
best_state = None

n_train = len(X_tr)
for epoch in range(1, n_epochs + 1):
    model_nn.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0

    for start in range(0, n_train, batch_size):
        bidx = perm[start:start + batch_size]
        x_b, y_b = X_tr[bidx], y_tr[bidx]

        optimizer.zero_grad()
        loss = loss_fn(model_nn(x_b), y_b)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(bidx)

    train_losses.append(epoch_loss / n_train)

    model_nn.eval()
    with torch.no_grad():
        val_logits = model_nn(X_vl)
        val_loss = loss_fn(val_logits, y_vl).item()
        val_acc = (val_logits.argmax(1) == y_vl).float().mean().item()
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        best_state = {k: v.detach().clone() for k, v in model_nn.state_dict().items()}
    else:
        no_improve += 1

    if epoch % 25 == 0:
        print(f"  Epoch {epoch:4d}  train_loss={train_losses[-1]:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model_nn.load_state_dict(best_state)
print(f"Best val loss: {best_val_loss:.4f}")

# plot training curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="train")
axes[0].plot(val_losses, label="val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy loss"); axes[0].set_title("Training/validation loss"); axes[0].legend()
axes[1].plot(val_accs);     axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy");           axes[1].set_title("Validation accuracy")
plt.tight_layout()
training_curve_path = os.path.join(OUTPUT_DIR, "approach3_training_curve.png")
plt.savefig(training_curve_path, dpi=150, bbox_inches="tight")
print(f"Training curve saved: {training_curve_path}")
plt.show()

# ── Step 4: predict all cells ─────────────────────────────────────────────────
model_nn.eval()
with torch.no_grad():
    probs = torch.softmax(model_nn(torch.tensor(X_all)), dim=1).numpy()

adata.obs["approach3_celltype"] = le.inverse_transform(probs.argmax(axis=1))
adata.obs["approach3_conf"]     = probs.max(axis=1)

print("\nApproach 3 prediction distribution:")
print(adata.obs["approach3_celltype"].value_counts())

# ── Step 5: visualize ─────────────────────────────────────────────────────────
# Check if UMAP exists; if not, compute it
if "X_umap" not in adata.obsm:
    print("X_umap not found in adata; computing UMAP for plotting...")
    sc.pp.neighbors(adata, use_rep="X_emb", n_neighbors=15)
    sc.tl.umap(adata)
    print("Computed adata X_umap for plotting")

fig, axes = plt.subplots(1, 3, figsize=(26, 6), constrained_layout=True)

sc.pl.umap(
    adata,
    color=CELLTYPIST_COL,
    title="Atlas labels (input)",
    ax=axes[0],
    show=False,
    legend_loc="right margin",
    legend_fontsize=7,
)

sc.pl.umap(
    adata,
    color="approach3_celltype",
    title="Approach 3 (NN)",
    ax=axes[1],
    show=False,
    legend_loc="right margin",
    legend_fontsize=7,
)

sc.pl.umap(
    adata,
    color="approach3_conf",
    title="Approach 3 confidence",
    ax=axes[2],
    show=False,
)

umap_path = os.path.join(OUTPUT_DIR, "approach3_umap.png")
plt.savefig(umap_path, dpi=180, bbox_inches="tight")
print(f"UMAP plot saved: {umap_path}")
plt.show()

# ── Step 6: save predictions ──────────────────────────────────────────────────
pred_cols = ["approach3_celltype", "approach3_conf"]
pred_path = os.path.join(OUTPUT_DIR, "approach3_predictions.csv")
adata.obs[pred_cols].to_csv(pred_path)
print(f"Prediction table saved: {pred_path}")

# ── Step 6.1: save updated AnnData ────────────────────────────────────────────
adata_h5ad_path = os.path.join(OUTPUT_DIR, "approach3_adata_reannotated.h5ad")
adata.write_h5ad(adata_h5ad_path)
print(f"Updated adata saved: {adata_h5ad_path}")

# ── Step 7: save the trained model ────────────────────────────────────────────
model_path = os.path.join(OUTPUT_DIR, "customnn_model.pt")
torch.save({
    "model_state":         model_nn.state_dict(),
    "label_classes":       le.classes_.tolist(),
    "celltypist_col_used": CELLTYPIST_COL,
    "n_input":             30,
    "n_classes":           n_classes,
}, model_path)
print(f"Model saved: {model_path}")

Cell 12 | markdown
---
## Part 1 · Final Comparison — All Three Approaches

Compare the three annotations side by side and build a consensus.

**Decision rules:**
- All three agree → high confidence, finalize label
- Two agree → moderate confidence, check marker genes
- All three disagree → manual review needed

In [ ]:
# Cell 13 | code
import scanpy as sc
import os
import matplotlib.pyplot as plt
# ── Load each approach's saved results and merge onto a common adata ─────────
# Each approach was run in its own kernel session reloading the base h5ad, so no
# single saved file already contains all three sets of predictions.
APPROACH1_DIR = str(data_path('cell_annotation_optimize/1.celltypist_retrain'))
APPROACH2_DIR = str(data_path('cell_annotation_optimize/2.customNB-VAE_scANVI_style'))
APPROACH3_DIR = str(data_path('cell_annotation_optimize/3.customNN_onXemb'))
FINAL_OUTPUT_DIR = str(data_path('cell_annotation_optimize'))

adata1 = sc.read_h5ad(os.path.join(APPROACH1_DIR, "approach1_reannotated.h5ad"))
adata2 = sc.read_h5ad(os.path.join(APPROACH2_DIR, "approach2_adata_reannotated.h5ad"))
adata3 = sc.read_h5ad(os.path.join(APPROACH3_DIR, "approach3_adata_reannotated.h5ad"))

# Approach 1's adata already carries the atlas labels/confidence and an X_umap —
# use it as the common backbone and attach Approach 2/3's predictions onto it.
adata = adata1.copy()
adata.obs["approach2_celltype"] = adata2.obs["approach2_celltype"].reindex(adata.obs_names)
adata.obs["approach2_conf"]     = adata2.obs["approach2_conf"].reindex(adata.obs_names)
adata.obs["approach3_celltype"] = adata3.obs["approach3_celltype"].reindex(adata.obs_names)
adata.obs["approach3_conf"]     = adata3.obs["approach3_conf"].reindex(adata.obs_names)

n_missing = adata.obs[["approach2_celltype", "approach3_celltype"]].isna().any(axis=1).sum()
if n_missing:
    print(f"Warning: {n_missing} cells missing an Approach 2/3 label after merge (check obs_names alignment)")
print(f"Merged adata: {adata.n_obs} cells, all 3 approaches present: {n_missing == 0}")

# Resolve the atlas label column locally (don't depend on Cell 4's in-memory state)
celltypist_label_cols = [c for c in adata.obs.columns if c.startswith("celltypist:") and not c.endswith(":conf")]
CELLTYPIST_COL = celltypist_label_cols[0]
CELLTYPIST_CONF_COL = CELLTYPIST_COL + ":conf"

# ── Side-by-side UMAP of all three approaches ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(30, 14), constrained_layout=True)

# Square each panel's plot box BEFORE plotting, so continuous-colorbar panels
# get their colorbar placed right next to the (already square) axes instead of
# the original wider footprint.
for ax in axes.flat:
    ax.set_box_aspect(1)

sc.pl.umap(adata, color=CELLTYPIST_COL,          title="Atlas (input)",           ax=axes[0,0], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="approach1_celltype",     title="Approach 1: CellTypist",  ax=axes[0,1], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="approach2_celltype",     title="Approach 2: scANVI",      ax=axes[0,2], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="approach3_celltype",     title="Approach 3: Custom NN",   ax=axes[1,0], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="approach1_conf",         title="Approach 1 confidence",   ax=axes[1,1], show=False)
sc.pl.umap(adata, color="approach3_conf",         title="Approach 3 confidence",   ax=axes[1,2], show=False)

# scanpy attaches continuous-value colorbars via make_axes_locatable, which
# constrained_layout doesn't manage — so the colorbar stays anchored to the
# pre-squeeze footprint. Run one layout pass, then snug the colorbars up to
# the now-square axes by hand, and freeze the layout so a later draw
# (savefig/show) doesn't let constrained_layout drift them apart again.
fig.canvas.draw()
cbar_axes = [ax for ax in fig.axes if ax not in axes.flat]
for main_ax, cbar_ax in zip([axes[1, 1], axes[1, 2]], cbar_axes):
    pos = main_ax.get_position()
    cbar_ax.set_position([pos.x1 + 0.005, pos.y0, 0.012, pos.height])
try:
    fig.set_layout_engine("none")
except AttributeError:
    fig.set_constrained_layout(False)

plt.suptitle("Cell Annotation — All Approaches", fontsize=14, y=1.02)
plt.savefig(os.path.join(FINAL_OUTPUT_DIR, "all_approaches_umap.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Per-cell consensus (no pre-existing Leiden cluster column is available) ──
from collections import Counter

def consensus_vote(row):
    votes = [row["approach1_celltype"], row["approach2_celltype"], row["approach3_celltype"]]
    most_common, count = Counter(votes).most_common(1)[0]
    if count == 3:
        return most_common, "all agree"
    elif count == 2:
        return most_common, "2/3 agree"
    return "review needed", "review needed"

# Keep the cell-type call and its agreement tier as separate columns — baking
# both into one string (e.g. "X (all agree)" vs "X (2/3 agree)") makes the
# same cell type show up as multiple different legend entries/colors.
votes_df = adata.obs.apply(consensus_vote, axis=1, result_type="expand")
adata.obs["consensus_celltype"]  = votes_df[0]
adata.obs["consensus_agreement"] = votes_df[1]

n_all_agree = (adata.obs["consensus_agreement"] == "all agree").sum()
n_majority  = (adata.obs["consensus_agreement"] == "2/3 agree").sum()
n_review    = (adata.obs["consensus_agreement"] == "review needed").sum()
print(f"\nConsensus (per cell, {adata.n_obs} total):")
print(f"  All 3 agree  : {n_all_agree}")
print(f"  2/3 agree    : {n_majority}")
print(f"  Review needed: {n_review}")

print("\nFinal consensus label distribution:")
print(adata.obs["consensus_celltype"].value_counts())

# ── Final UMAP: cell type and agreement tier as separate panels ──────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)
sc.pl.umap(adata, color="consensus_celltype",
           title="Consensus cell type",
           ax=axes[0], show=False,
           legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="consensus_agreement",
           title="Consensus agreement",
           ax=axes[1], show=False,
           legend_loc="right margin", legend_fontsize=8)
plt.savefig(os.path.join(FINAL_OUTPUT_DIR, "consensus_umap.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Save final annotated h5ad ─────────────────────────────────────────────────

final_h5ad_path = os.path.join(FINAL_OUTPUT_DIR, "annotated.h5ad")
print(f"Saved: {final_h5ad_path}")
adata.write_h5ad(final_h5ad_path)

Cell 14 | markdown
---
## Part 1 · Which Approach is the Best?

There is no single "best" — it depends on what you prioritize. Here is an honest ranking for this workflow.

---

### Recommended order of use

#### Start with Approach 2 (scANVI) — best overall when cell types form a continuum

**Why:**
- Many cell types exist on a **continuum** (e.g. subtypes grade into each other) rather than discrete clusters. scANVI's latent space captures this gradient; logistic regression (Approach 1) draws hard linear boundaries
- Handles your low-confidence cells natively — marks them `"Unknown"` and infers their type from surrounding cells in the latent space
- The new `X_scANVI` embedding produces a **biologically cleaner UMAP** where cell types separate more distinctly than the raw scVI UMAP
- Directly integrates with your existing scVI model — no extra gene expression preprocessing needed

**When it can fail:** if your preliminary atlas labels are very noisy (e.g. many low-confidence cells), the supervised signal is weak and you may get poor results. Check that you have ≥ 50 labeled cells per cell type before training.

---

#### Then validate with Approach 1 (CellTypist retrain) — best for marker discovery

**Why:**
- Gives you an **explicit ranked list of marker genes** per cell type — directly usable for publication figures (`dotplot`, `violin`) and validation
- Fast to run and easy to interpret
- The learned coefficients tell you which genes drive each cell type call in YOUR data (not the reference atlas)

**Use it as a second pass** to confirm Approach 2 calls and to extract the gene-level evidence behind each cluster's annotation.

---

#### Use Approach 3 only if you have a specific reason

**Why it is the weakest default choice:**
- Trains on the 30-dim `X_emb` only — gives no marker genes, no embedding improvement
- Approach 2 (scANVI) is strictly better as a deep learning classifier because it jointly optimizes the latent space AND the classifier
- Useful if you want a **fast, portable model** to run on new samples without reinstalling scvi-tools (just `torch` + saved `.pt` weights)

---

### Summary

| Priority | Best approach |
|----------|--------------|
| Best biology, clean UMAP, handles unknowns | **Approach 2 (scANVI)** |
| Explicit marker genes for publication | **Approach 1 (CellTypist retrain)** |
| Fast, portable model for new samples | **Approach 3 (custom NN)** |
| Most defensible final annotation | **Run all three + take consensus** |

The practical recommendation is: **run Approach 2 first, use Approach 1 to extract the markers, and use the consensus from the final comparison cell to finalize `adata.obs["consensus_celltype"]`**.

Cell 15 | markdown
---
# ══════════  PART 2  ════════
### Reference-Free & Limited-Atlas Methods
*(no trustworthy atlas required — for any tissue)*

The three approaches above are all **reference-based supervised classifiers**: each is trained on the
preliminary CellTypist labels, so all three inherit whatever those labels get wrong. For a tissue with a
**limited, noisy, or mismatched atlas** (non-model organism, rare organ, novel disease state) this is the
ceiling on every result — the models will *force-align* novel cell types to the nearest known class with
false confidence.

**This section adds methods that do not depend on a trustworthy atlas**, plus a reject option so
out-of-reference cells become `"Unknown"` instead of a confident wrong label.

| Method | Category | Needs a reference? | Handles novel types |
|---|---|---|---|
| Reject option | uncertainty gate | — | ✓ flags them |
| scType | marker-based scoring | ✗ (marker list only) | ✓ (unknown guard) |
| AUCell | rank-based scoring | ✗ (marker list only) | partial |
| Marker discovery (NS-Forest / Tau / COSG) | de-novo markers | ✗ (query only) | n/a (finds markers) |
| Zero-shot centroid | foundation-model embedding | few anchors | partial |

> **Note on foundation models (scGPT / Geneformer / UCE):** 2025 benchmarks (Kedzierska et al., *Genome
> Biology*) found their **zero-shot** cell-type separation is often *worse* than scVI / HVG+PCA / Harmony.
> Use them as an alternative embedding and cross-check — do **not** trust raw zero-shot labels.

Cell 16 | markdown
## Part 2.1 — Reject option (turn low-confidence calls into `"Unknown"`)

Any softmax classifier always emits a label. Convert calls below a confidence threshold (or with high
prediction entropy) to `"Unknown"` — these become the candidates for a novel population.

In [ ]:
# Cell 17 | code
import numpy as np
import pandas as pd

CONF_REJECT = 0.5   # ← tune per dataset (probability below this ⇒ Unknown)

# Approach 1 and Approach 3 expose a per-cell max-probability confidence.
for label_col, conf_col in [("approach1_celltype", "approach1_conf"),
                            ("approach3_celltype", "approach3_conf")]:
    if label_col in adata.obs and conf_col in adata.obs:
        rej = label_col + "_rej"
        adata.obs[rej] = adata.obs[label_col].astype(str)
        adata.obs.loc[adata.obs[conf_col] < CONF_REJECT, rej] = "Unknown"
        n_unk = (adata.obs[rej] == "Unknown").sum()
        print(f"{rej:28s}: {n_unk:6d} cells -> Unknown")

# scANVI (Approach 2) supports Unknown natively via its unlabeled_category.
print("\\nApproach 2 (scANVI) handles unknowns during training via unlabeled_category='Unknown'.")

Cell 18 | markdown
## Part 2.2 — scType (reference-free, marker-based)

For each marker gene, a **specificity weight** down-weights genes shared by many cell types:

$$S_g = 1 - \frac{N_g - \min N}{\max N - \min N}$$

where $N_g$ = number of cell types that list gene $g$. Then the (log-normalized, z-scored) expression is
weighted by $S$ and summed over each type's markers; the cluster is assigned its argmax, with a guard that
flags scores below `n_cells/4` as `"Unknown"`.

**Replace `marker_dict` with markers for YOUR tissue's cell types** — from literature, a related tissue, or
`marker_discovery()` (II.4).

In [ ]:
# Cell 19 | code
from collections import Counter

def compute_sctype(adata, marker_dict, cluster_col="scvi_leiden"):
    """scType-style reference-free annotation.
    marker_dict: {cell_type: [positive genes]}  or  {cell_type: {"positive":[...], "negative":[...]}}
    Assumes adata.X is log-normalized."""
    md_ = {ct: (m if isinstance(m, dict) else {"positive": list(m), "negative": []})
           for ct, m in marker_dict.items()}
    # 1. gene specificity weights (over positive markers)
    cnt = Counter(g for m in md_.values() for g in m["positive"])
    Ng = dict(cnt); mn, mx = min(Ng.values()), max(Ng.values())
    S = {g: (1 - (Ng[g] - mn) / (mx - mn)) if mx > mn else 1.0 for g in Ng}
    # 2. z-score the marker genes across cells
    X = adata.to_df()
    all_markers = sorted({g for m in md_.values() for g in (m["positive"] + m["negative"])} & set(X.columns))
    Z = (X[all_markers] - X[all_markers].mean()) / (X[all_markers].std() + 1e-9)
    # 3. per-cell type score = Σ S·z (positive) − Σ S·z (negative)
    scores = pd.DataFrame(index=adata.obs_names)
    for ct, m in md_.items():
        pos = [g for g in m["positive"] if g in Z.columns]
        neg = [g for g in m["negative"] if g in Z.columns]
        s = sum(S.get(g, 1.0) * Z[g] for g in pos) if pos else pd.Series(0.0, index=Z.index)
        if neg:
            s = s - sum(S.get(g, 1.0) * Z[g] for g in neg)
        scores[ct] = s
    # 4. aggregate per cluster, argmax, unknown-guard
    scores[cluster_col] = adata.obs[cluster_col].values
    agg = scores.groupby(cluster_col).sum()
    n_per = adata.obs[cluster_col].value_counts()
    calls = {cl: (agg.loc[cl].idxmax() if agg.loc[cl].max() >= n_per[cl] / 4 else "Unknown")
             for cl in agg.index}
    adata.obs["sctype_celltype"] = adata.obs[cluster_col].map(calls).astype("category")
    return adata.obs["sctype_celltype"], agg

# ── EXAMPLE marker dict — REPLACE with your tissue's markers ───────────────────
marker_dict = {
    "Kenyon_cell":       ["ey", "Dop1R2", "prt", "CrebA"],
    "Glia":              ["repo", "nrv2", "Gs2", "alrm"],
    "Optic_lobe_neuron": ["Optix", "dac", "so"],
    # ... add one entry per expected cell type
}
sctype_labels, sctype_agg = compute_sctype(adata, marker_dict)
print("scType annotation (per cluster, reference-free):")
print(adata.obs["sctype_celltype"].value_counts())

Cell 20 | markdown
## Part 2.3 — AUCell (reference-free, rank-based)

For each cell, rank all genes (1 = highest). A marker set scores high if its genes concentrate near the
**top** of the ranking. AUCell measures the area under the marker-recovery curve over the top *k* ranks,
normalized to [0, 1]. Because it uses only ranks, it is invariant to how the data were normalized.

In [ ]:
# Cell 21 | code
def compute_aucell(adata, marker_dict, max_rank_frac=0.05):
    """AUCell-style rank enrichment. Returns a (cells × cell_types) AUC DataFrame."""
    X = adata.to_df()
    n_genes = X.shape[1]
    k = max(1, int(n_genes * max_rank_frac))
    ranks = X.rank(axis=1, ascending=False, method="first").values  # 1 = highest
    gene_idx = {g: i for i, g in enumerate(X.columns)}
    auc = pd.DataFrame(index=adata.obs_names)
    for ct, m in marker_dict.items():
        genes = m["positive"] if isinstance(m, dict) else m
        cols = [gene_idx[g] for g in genes if g in gene_idx]
        if not cols:
            continue
        mset = len(cols)
        r = ranks[:, cols]                       # cells × m marker ranks
        area = np.zeros(len(X))
        for x in range(1, k + 1):
            area += (r <= x).sum(1)              # recovery(x) summed over ranks
        max_area = (k * mset - mset * (mset - 1) / 2) if mset <= k else k * (k + 1) / 2
        auc[ct] = area / max_area
    return auc

aucell = compute_aucell(adata, marker_dict)
adata.obs["aucell_celltype"] = aucell.idxmax(axis=1).values
adata.obs["aucell_conf"]     = aucell.max(axis=1).values
print("AUCell annotation (per cell, reference-free):")
print(adata.obs["aucell_celltype"].value_counts())

Cell 22 | markdown
## Part 2.4 — Reference-free marker-gene discovery (NS-Forest / Tau / COSG)

The inverse of II.2: **discover** the markers that define each Leiden cluster from the query data alone —
this is how you *build* a marker list for a tissue no atlas covers.

- **Binary Expression Score** (NS-Forest): rewards genes expressed in the target cluster and silent elsewhere:
  $$\mathrm{BES}_{g,T} = \frac{1}{n-1}\sum_{i\neq T}\left(1 - \frac{y_i}{y_T}\right)^{+},\qquad y_i=\text{median expr in cluster }i$$
- **Tau** specificity: $\tau = \frac{1}{n-1}\sum_i (1-\hat x_i),\ \hat x_i = x_i/\max_i x_i$
- **COSG**: cosine similarity between a gene's expression and the cluster indicator.

Feed the discovered panels straight back into `compute_sctype()` for a fully atlas-free annotation loop.

In [ ]:
# Cell 23 | code
def marker_discovery(adata, cluster_col="scvi_leiden", top_n=10):
    """Reference-free per-cluster markers via Binary Expression Score, Tau, and COSG."""
    X = adata.to_df()
    clusters = adata.obs[cluster_col].astype(str)
    cl_ids = sorted(clusters.unique())
    n = len(cl_ids)
    med  = pd.DataFrame({cl: X[clusters == cl].median() for cl in cl_ids})
    mean = pd.DataFrame({cl: X[clusters == cl].mean()   for cl in cl_ids})
    xmax = mean[cl_ids].max(1)
    tau  = (1 - mean[cl_ids].div(xmax, axis=0)).sum(1) / (n - 1)   # global gene specificity
    Xv, Xnorm = X.values, np.linalg.norm(X.values, axis=0) + 1e-9
    out = {}
    for cl in cl_ids:
        yT = med[cl].replace(0, np.nan)
        bes = sum(np.clip(1 - med[c] / yT, 0, None) for c in cl_ids if c != cl) / (n - 1)
        bes = bes.fillna(0)
        ind = (clusters == cl).astype(float).values
        cosg = pd.Series((Xv.T @ ind) / (Xnorm * (np.linalg.norm(ind) + 1e-9)), index=X.columns)
        df = pd.DataFrame({"BES": bes, "Tau": tau, "COSG": cosg}).fillna(0)
        df["score"] = df["BES"] * df["COSG"]                       # BES-primary, COSG tiebreak
        out[cl] = df.sort_values("score", ascending=False).head(top_n)
    return out

discovered = marker_discovery(adata)
print("Top de-novo markers per Leiden cluster (reference-free):")
for cl, df in list(discovered.items())[:5]:
    print(f"  cluster {cl}: {df.index.tolist()}")

# turn discovered panels into a scType-ready marker_dict (keyed by cluster id):
discovered_marker_dict = {f"cluster_{cl}": df.index.tolist() for cl, df in discovered.items()}

Cell 24 | markdown
## Part 2.5 — (Optional) Foundation-model zero-shot via nearest centroid

If you have a foundation-model embedding (scGPT / Geneformer / UCE / scFoundation) in `adata.obsm`, you can
transfer labels from a few anchors by cosine-nearest centroid. Here we default to the scVI `X_emb`; swap in a
foundation embedding to compare.

**Caveat (2025 benchmarks):** zero-shot foundation-model labels are frequently *worse* than scVI/HVG. Use
this only to cross-check, and validate against the marker-based scores above.

In [ ]:
# Cell 25 | code
def zero_shot_centroid(adata, embed_key="X_emb",
                       label_col=CELLTYPIST_COL, conf_col=CELLTYPIST_CONF_COL,
                       thresh=CONF_THRESHOLD):
    """Nearest-centroid label transfer in an embedding (L2-normalized cosine)."""
    E = np.asarray(adata.obsm[embed_key], dtype=float)
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
    labeled = (adata.obs[conf_col] > thresh).values
    cents = {}
    for ct in adata.obs.loc[labeled, label_col].unique():
        m = labeled & (adata.obs[label_col] == ct).values
        if m.sum() < 5:            # need a few anchors per type
            continue
        c = E[m].mean(0); cents[ct] = c / (np.linalg.norm(c) + 1e-9)
    cts = list(cents); C = np.vstack([cents[ct] for ct in cts])
    sim = E @ C.T
    adata.obs["zeroshot_celltype"] = pd.Categorical([cts[i] for i in sim.argmax(1)])
    adata.obs["zeroshot_conf"]     = sim.max(1)
    return adata.obs["zeroshot_celltype"]

# swap embed_key="X_scGPT" (or your foundation embedding) once available
zero_shot_centroid(adata, embed_key="X_emb")
print("Zero-shot (nearest-centroid) distribution:")
print(adata.obs["zeroshot_celltype"].value_counts())

Cell 26 | markdown
## Part 2.6 — Cross-family consensus (supervised + marker-based)

For a limited-atlas tissue, the defensible annotation is a **consensus across method *families*** — not
across three supervised models that all read the same seed labels. Combine one supervised call (scANVI) with
two reference-free calls (scType, AUCell), apply the reject option, and flag disagreement as review.

In [ ]:
# Cell 27 | code
from collections import Counter

def cross_family_consensus(row, cols):
    votes = [str(row[c]) for c in cols if c in row and str(row[c]) not in ("nan", "Unknown")]
    if not votes:
        return "Unknown"
    label, count = Counter(votes).most_common(1)[0]
    return label if count >= 2 else "review needed"

family_cols = [c for c in ["approach2_celltype", "sctype_celltype", "aucell_celltype"] if c in adata.obs]
adata.obs["consensus_v2"] = adata.obs.apply(lambda r: cross_family_consensus(r, family_cols), axis=1)

print(f"Cross-family consensus over {family_cols}:")
print(adata.obs["consensus_v2"].value_counts())

# cells flagged for review = the real scientific findings (candidate novel populations)
n_review = (adata.obs["consensus_v2"] == "review needed").sum()
n_unknown = (adata.obs["consensus_v2"] == "Unknown").sum()
print(f"\\n{n_review} cells need review, {n_unknown} Unknown — inspect these for novel cell types.")

adata.write_h5ad("annotated_v2.h5ad")
print("Saved: annotated_v2.h5ad")

Cell 28 | markdown
## Part 2.7 — Decision guide (any tissue, limited atlas)

| Situation | Do this | Cells above |
|---|---|---|
| Good matching reference (≥ 50 cells/type) | scANVI → validate with CellTypist markers → consensus | Part I |
| Thin / noisy atlas | add the **reject option**; cross-check with **scType** or **AUCell** | II.1–II.3 |
| Marker list but no reference data | **scType / AUCell** directly | II.2–II.3 |
| No markers either | **marker_discovery()** (NS-Forest/Tau/COSG) → then scType | II.4 → II.2 |
| Non-model organism, no conspecific atlas | cross-species transfer: **SAMap / SATURN / CAME** (R/Python, not shown) | — |

**One habit that survives every case:** run ≥ 1 reference-dependent method and ≥ 1 reference-free method,
apply a reject option, and treat the cells where they disagree as candidate novel populations — not noise.